In [ ]:
import json

# Supervised

In [ ]:
RF_RESULTS_ROOT_PATH = r"C:\Users\neosw\Desktop\WL\V6_supervised_features\_rf_results"

# read in ALL_summaries.json
with open(f"{RF_RESULTS_ROOT_PATH}/ALL_summaries.json", "r") as f:
    all_summaries = json.load(f)

In [ ]:
all_summaries


In [ ]:
import pandas as pd

# Initialize an empty list to store rows
rows = []

# Iterate through all_summaries
for summary in all_summaries:
    tag = summary["tag"]
    intermediate_results = summary["intermediate_results"]

    # Iterate through each intermediate result
    for result in intermediate_results:
        # Extract the parameters and f1_score
        params = result[0]
        f1_score = result[1]

        # Create a row with the tag, parameters, and f1_score
        row = {"tag": tag, **params, "f1_score": f1_score}
        rows.append(row)

# Create a DataFrame from the rows
df_intermediate_results = pd.DataFrame(rows)

# Display the DataFrame
df_intermediate_results

In [ ]:
def format_tag(tag):
    # Split the tag into parts
    parts = tag.split("_")
    W_feat = f"{int(parts[0][:-1])}min"

    # Map the second part to the corresponding W_label
    W_label_map = {"no": "0h", "11163": "3.1h", "22326": "6.2h"}
    W_label = W_label_map.get(parts[1], "Unknown")

    # Format the tag
    return f"W_feat: {W_feat} | W_label: {W_label}"


# Apply the formatting function to the tag column
df_intermediate_results["tag"] = df_intermediate_results["tag"].apply(format_tag)

# Display the updated DataFrame
df_intermediate_results

In [ ]:
# Sort the DataFrame by 'tag' and 'f1_score' in descending order
df_intermediate_results = df_intermediate_results.sort_values(by=["tag", "f1_score"], ascending=[True, False])

# Assign an internal ID per tag
df_intermediate_results["id"] = df_intermediate_results.groupby("tag").cumcount() + 1

# Display the updated DataFrame
df_intermediate_results


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = df_intermediate_results.copy()

# --- Split tag into left (before |) and right (after |) ---
lr = df["tag"].str.split(r"\s*\|\s*", n=1, expand=True)
df["tag_left"] = lr[0].str.strip()
df["tag_right"] = lr[1].str.strip()

# Desired fixed orders
ahc_order = ["W_label: 0h", "W_label: 3.1h", "W_label: 6.2h"]  # subplot order (left→right)
left_order_norm = ["W_feat: 6min", "W_feat: 11min", "W_feat: 23min"]  # y-axis order (bottom→top)

# Enforce categorical order for y (bottom→top is the default; don't invert y)
df["tag_left"] = pd.Categorical(df["tag_left"], categories=left_order_norm, ordered=True)

# --- Plot: 3 side-by-side subplots, one per requested AHC ---
n = len(ahc_order)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 6), dpi=300, sharey=True)

# Ensure axes is iterable even if n==1
if n == 1:
    axes = [axes]

for ax, ahc in zip(axes, ahc_order, strict=False):
    sub = df[df["tag_right"] == ahc]

    # Horizontal boxplot of F1 grouped by LEFT part (normalized for order)
    sub.boxplot(
        column="f1_score",
        by="tag_left",
        grid=False,
        vert=False,
        ax=ax,
    )

    ax.set_xlim(0, 0.6)  # F1 score range

    # Clean up titles/grid
    ax.set_title(ahc)
    ax.set_xlabel("Mean (F1 Score)")
    ax.set_ylabel("")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="x", linestyle="--", alpha=0.7)

# Remove pandas' automatic super title
plt.suptitle("")
plt.tight_layout()
plt.show()


In [ ]:
from typing import Tuple

import numpy as np
from scipy.signal import butter, filtfilt
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from tqdm import tqdm


# Define a low-pass filter
def low_pass_filter(data, cutoff, fs, order=4):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype="low", analog=False)
    filtered_data = filtfilt(b, a, data)
    return filtered_data


def evaluate_at_best_threshold_and_cutoff(
    y_true: np.ndarray, y_proba: np.ndarray, fs: int = 1
) -> Tuple[float, float, float, float, int, float]:
    """
    Search for the best combination of threshold and cutoff frequency maximizing F1 (for class 1).
    Returns (best_thr, best_f1, best_p, best_r, support_pos, best_cutoff).
    """
    thresholds = np.linspace(0.01, 0.99, 99)
    cutoff_frequencies = np.linspace(0.01, 0.1, 99)  # Range of cutoff frequencies to test
    best_f1, best_thr, best_cutoff, best_p, best_r = -1.0, 0.5, 0.01, 0.0, 0.0
    for cutoff in tqdm(cutoff_frequencies, desc="Evaluating cutoff frequencies"):
        # Apply low-pass filter
        filtered_proba = low_pass_filter(y_proba, cutoff, fs)
        for thr in thresholds:
            y_pred = (filtered_proba >= thr).astype(int)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_thr, best_cutoff = f1, float(thr), float(cutoff)
                best_p = precision_score(y_true, y_pred, zero_division=0)
                best_r = recall_score(y_true, y_pred, zero_division=0)

    return best_thr, best_f1, best_p, best_r, int((y_true == 1).sum()), best_cutoff


In [ ]:
import os

# Initialize a list to store the results
results = []

# Iterate over files in the RF_RESULTS_ROOT_PATH
# Iterate over files in the RF_RESULTS_ROOT_PATH with a progress bar
for file_name in tqdm(os.listdir(RF_RESULTS_ROOT_PATH), desc="Processing files"):
    if file_name.endswith("_scores.json"):
        # Extract W_feat and W_label from the file name
        parts = file_name.split("_")
        W_feat = parts[0]
        W_label = parts[1]

        # Load the JSON file
        with open(os.path.join(RF_RESULTS_ROOT_PATH, file_name), "r") as f:
            data = json.load(f)

        # Extract y_true and y_proba from the dev_eval object
        y_true = data["dev_eval"]["labels"]
        y_proba = data["dev_eval"]["scores"]

        # Evaluate at the best threshold and cutoff
        best_thr, best_f1, best_p, best_r, _, best_cutoff = evaluate_at_best_threshold_and_cutoff(
            y_true=np.array(y_true), y_proba=np.array(y_proba)
        )

        # Append the results to the list
        results.append(
            {
                "W_feat": W_feat,
                "W_label": W_label,
                "f1_score": best_f1,
                "precision": best_p,
                "recall": best_r,
                "best_cutoff": best_cutoff,
                "threshold": best_thr,
            }
        )

# Create a DataFrame from the results
df_scores = pd.DataFrame(results)


In [ ]:
# Order the DataFrame by 'f1_score' in descending order
df_scores = df_scores.sort_values(by="f1_score", ascending=False)

# Display the DataFrame
df_scores

In [ ]:
from pprint import pprint

# Filter the all_summaries list for the specific tag
best = next((summary for summary in all_summaries if summary["tag"] == "6m_22326_shift"), None)

# Display the result
# Nicely print 'best_params' and 'test' sections
pprint(best["best_params"])

In [ ]:
len(feature_importances)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.tree import plot_tree

# Extract the RandomForestClassifier from the pipeline
rf_model = model.named_steps["clf"]

# Extract feature importances
feature_importances = rf_model.feature_importances_
feature_names = model.named_steps["pre"].transformers_[0][2]  # Extract feature names from the pipeline

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": feature_importances})
importance_df = importance_df.sort_values(by="Importance", ascending=False).head(15)

# Plot feature importances
plt.figure(figsize=(10, 6), dpi=300)
plt.barh(importance_df["Feature"], importance_df["Importance"], color="skyblue")
plt.xlabel("Importance")
plt.gca().invert_yaxis()  # Invert y-axis to show the most important feature at the top
plt.grid(axis="x", linestyle="--", alpha=0.7)

# Remove top and right spines
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

# Visualize one of the trees in the RandomForest
# Extract the first tree from the RandomForestClassifier
tree = rf_model.estimators_[0]

# Plot the tree
plt.figure(figsize=(38, 10), dpi=300)
plot_tree(
    tree,
    feature_names=feature_names,
    filled=True,
    rounded=True,
    proportion=True,
    fontsize=10,
)
plt.title("1st Decision Tree from Random Forest")
plt.show()


In [ ]:
# Load the JSON file
with open(r"C:\Users\neosw\Desktop\WL\V6_supervised_features\_rf_results\6m_22326_shift_scores.json", "r") as f:
    scores = json.load(f)

In [ ]:
thr = 0.15
best_cutoff = 0.012755
test_scores = scores["test"]["scores"]
true_labels = scores["test"]["labels"]

In [ ]:
import polars as pl

TIME_COL = "timestamp"
LABEL_COL = "window_end"


def read_features_parquet(path: str) -> pl.DataFrame:
    df = pl.read_parquet(path)
    if TIME_COL in df.columns:
        df = df.with_columns(pl.col(TIME_COL).cast(pl.Datetime))
    if LABEL_COL not in df.columns:
        raise ValueError(f"Expected '{LABEL_COL}' in {path} but not found.")
    return df.sort(TIME_COL)


In [ ]:
sup_test_data = read_features_parquet(
    r"C:\Users\neosw\Desktop\WL\V6_supervised_features\6m_22326_shift\test.parquet"
).to_pandas()

sup_test_data["sanity_label"] = true_labels
sup_test_data["predicted_score"] = low_pass_filter(np.array(test_scores), best_cutoff, fs=1)

In [ ]:
# Check for mismatched labels
mismatched_labels = sup_test_data[sup_test_data["label"] != sup_test_data["sanity_label"]]

# Display the mismatched rows, if any
if not mismatched_labels.empty:
    print("Mismatched labels found:")
    print(mismatched_labels)
else:
    print("No mismatched labels found.")

In [ ]:
failure_data = pd.read_csv(r"C:\Users\neosw\Desktop\WL\asset_data\failure.csv", sep=";")

In [ ]:
from datetime import datetime

import pandas as pd

# Load the failure data
failure_data = pd.read_csv(r"C:\Users\neosw\Desktop\WL\asset_data\failure.csv", sep=";")

# Combine 'Störungsbeginn' and 'Störungsbeginn Zeit' into a single datetime column
failure_data["Start"] = pd.to_datetime(
    failure_data["Störungsbeginn"] + " " + failure_data["Störungsbeginn Zeit"], format="%d.%m.%Y %H:%M:%S"
)

# Combine 'Störungsende' and 'Störungsende Zeit' into a single datetime column
failure_data["End"] = pd.to_datetime(
    failure_data["Störungsende"] + " " + failure_data["Störungsende Zeit"], format="%d.%m.%Y %H:%M:%S"
)

# filter to have end after 2025-02-10
failure_data = failure_data[failure_data["End"] > datetime(2025, 2, 10)]

# filtler failure_data to have start before 2025-06-01
failure_data = failure_data[failure_data["Start"] < datetime(2025, 6, 1)]

# create a Shifted_Start and Shifted_End where vlaue is shifted 22326 seconds into the past
failure_data["Shifted_Start"] = failure_data["Start"] - pd.Timedelta(seconds=22326)
failure_data["Shifted_End"] = failure_data["End"] - pd.Timedelta(seconds=22326)

In [ ]:
from sklearn.metrics import f1_score

# Binarize the predicted_score column using the threshold
sup_test_data["predicted_label"] = (sup_test_data["predicted_score"] >= thr).astype(int)

# Calculate metrics
f1 = f1_score(sup_test_data["label"], sup_test_data["predicted_label"])
precision = precision_score(sup_test_data["label"], sup_test_data["predicted_label"])
recall = recall_score(sup_test_data["label"], sup_test_data["predicted_label"])
conf_matrix = confusion_matrix(sup_test_data["label"], sup_test_data["predicted_label"])

# Display the results
print(f"F1 Score: {f1}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print("Confusion Matrix:")
print(conf_matrix)

In [ ]:
from datetime import timedelta

# Initialize counters
detected_failures = 0
false_alarms = 0
thr = 0.15

# order sup_test_data by window_end
sup_test_data = sup_test_data.sort_values(by="window_end")

# Iterate through the failure data
for _, failure in failure_data.iterrows():
    failure_start = failure["Start"]
    failure_end = failure["End"]

    # Define the 4-day window around the failure
    window_start = failure_start - timedelta(days=6)
    window_end = failure_start - timedelta(days=1)

    # Filter test_data within the 4-day window
    window_data = sup_test_data[
        (sup_test_data["window_end"] >= window_start) & (sup_test_data["window_end"] <= window_end)
    ]

    # Check for continuous crossings above the threshold
    above_threshold = window_data["predicted_score"] >= thr
    if above_threshold.any():
        # Count as detected if it stays above the threshold at least once
        detected_failures += 1

# Check for false alarms (crossings outside of failure windows)
for idx, row in sup_test_data.iterrows():
    timestamp_end = row["window_end"]
    timestamp_start = row["timestamp"]
    predicted_score = row["predicted_score"]

    # If the predicted_score crosses the threshold
    if predicted_score >= thr:
        # Check if the timestamp is within any failure window
        in_failure_window = any(
            (timestamp_start <= failure["End"] and timestamp_end >= failure["Start"] - timedelta(days=6))
            for _, failure in failure_data.iterrows()
        )

        # If not in a failure window, check for crossing behavior
        if not in_failure_window:
            # Check if the previous score was below the threshold using the index
            if idx > 0:  # Ensure there is a previous row
                previous_row = sup_test_data.iloc[idx - 1]
                if previous_row["predicted_score"] < thr:
                    false_alarms += 1

# Display the results
print(f"Detected Failures: {detected_failures}")
print(f"False Alarms: {false_alarms}")


In [ ]:
from datetime import timedelta

# Initialize counters
detected_failures = 0
false_alarms = 0
thr = 0.15
# Define the cutoff date
cutoff_date = pd.Timestamp("2025-04-15")

# order sup_test_data by window_end
sup_test_data = sup_test_data.sort_values(by="window_end")

# Iterate through the failure data
for _, failure in failure_data.iterrows():
    failure_start = failure["Start"]
    failure_end = failure["End"]

    # Skip failures after the cutoff date
    if failure_start > cutoff_date:
        continue

    # Define the 4-day window around the failure
    window_start = failure_start - timedelta(days=6)
    window_end = failure_start - timedelta(days=1)

    # Filter test_data within the 4-day window
    window_data = sup_test_data[
        (sup_test_data["window_end"] >= window_start) & (sup_test_data["window_end"] <= window_end)
    ]

    # Check for continuous crossings above the threshold
    above_threshold = window_data["predicted_score"] >= thr
    if above_threshold.any():
        # Count as detected if it stays above the threshold at least once
        detected_failures += 1

# Check for false alarms (crossings outside of failure windows)
for idx, row in sup_test_data.iterrows():
    timestamp_end = row["window_end"]
    timestamp_start = row["timestamp"]
    predicted_score = row["predicted_score"]

    # Skip rows after the cutoff date
    if timestamp_end > cutoff_date:
        continue

    # If the predicted_score crosses the threshold
    if predicted_score >= thr:
        # Check if the timestamp is within any failure window
        in_failure_window = any(
            (timestamp_start <= failure["End"] and timestamp_end >= failure["Start"] - timedelta(days=6))
            for _, failure in failure_data.iterrows()
        )

        # If not in a failure window, check for crossing behavior
        if not in_failure_window:
            # Check if the previous score was below the threshold using the index
            if idx > 0:  # Ensure there is a previous row
                previous_row = sup_test_data.iloc[idx - 1]
                if previous_row["predicted_score"] < thr:
                    false_alarms += 1

# Display the results
print(f"Detected Failures: {detected_failures}")
print(f"False Alarms: {false_alarms}")


In [ ]:
from datetime import datetime

import matplotlib.dates as mdates
import matplotlib.pyplot as plt


def plot_predicted_scores(data: pd.DataFrame, xlim: tuple[datetime, datetime], threshold: float) -> None:
    """
    Plots predicted scores over time with an adjustable x-axis limit and threshold.
    Returns a DataFrame with predicted scores.

    Parameters:
    xlim (tuple): A tuple of two datetime objects specifying the x-axis limits.
    threshold (float): The threshold value for the horizontal line.
    """
    # Create the plot
    fig, ax = plt.subplots(figsize=(12, 6))

    # Plot predicted_score over time
    ax.plot(data["window_end"], data["predicted_score"], label="Predicted Score", color="blue")

    # Add a horizontal line at the threshold
    ax.axhline(y=threshold, color="black", linestyle="--", label=f"Threshold ({threshold})")

    # Format x-axis as time
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    plt.xticks(rotation=45)

    # Add shaded regions for failure zones
    for _, row in failure_data.iterrows():
        ax.axvline(x=row["Start"] - pd.Timedelta(days=1), color="green", linestyle="--", label="Time to fix Failure")
        ax.axvspan(row["Start"], row["End"], color="red", alpha=0.3, label="Failure Zone")

    # Remove duplicate labels in the legend
    handles, labels = ax.get_legend_handles_labels()
    unique_labels = dict(zip(labels, handles, strict=False))
    ax.legend(unique_labels.values(), unique_labels.keys())

    # Set x-axis limits
    ax.set_xlim(xlim)

    # Remove top and right spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Add labels, legend, and grid
    ax.set_xlabel("Time")
    ax.set_ylabel("Predicted Score")
    plt.grid(axis="y", linestyle="--", alpha=0.7)

    # Show the plot
    plt.tight_layout()
    plt.show()


In [ ]:
plot_predicted_scores(sup_test_data, (datetime(2025, 2, 10), datetime(2025, 6, 2)), thr)

In [ ]:
for failure_start, subsystem in zip(failure_data["Start"], failure_data["Subsystem Bezeichner"], strict=False):
    print(f"Subsystem Bezeichner: {subsystem}")
    start_xlim = failure_start - pd.Timedelta(days=6)
    end_xlim = failure_start + pd.Timedelta(days=1)
    plot_predicted_scores(sup_test_data, (start_xlim, end_xlim), thr)

# Unsupervised

In [ ]:
LSTMAE__RESULTS_ROOT_PATH = r"C:\Users\neosw\Desktop\WL\V6_unsupervised_sequences\_ae_results"

In [ ]:
import os

# Initialize a list to store the results
results = []

# Iterate over files in the LSTMAE__RESULTS_ROOT_PATH
# Iterate over files in the LSTMAE__RESULTS_ROOT_PATH with a progress bar
for file_name in tqdm(os.listdir(LSTMAE__RESULTS_ROOT_PATH), desc="Processing files"):
    if file_name.endswith("_scores.json"):
        # Extract W_feat and W_label from the file name
        parts = file_name.split("_")
        W_feat = parts[0]
        W_label = parts[1]

        # Load the JSON file
        with open(os.path.join(LSTMAE__RESULTS_ROOT_PATH, file_name), "r") as f:
            data = json.load(f)

        # Extract y_true and y_proba from the dev_eval object
        y_true = data["dev_eval"]["labels"]
        y_proba = data["dev_eval"]["scores"]

        # Evaluate at the best threshold and cutoff
        best_thr, best_f1, best_p, best_r, _, best_cutoff = evaluate_at_best_threshold_and_cutoff(
            y_true=np.array(y_true), y_proba=np.array(y_proba)
        )

        # Append the results to the list
        results.append(
            {
                "W_feat": W_feat,
                "W_label": W_label,
                "f1_score": best_f1,
                "precision": best_p,
                "recall": best_r,
                "best_cutoff": best_cutoff,
                "threshold": best_thr,
            }
        )

# Create a DataFrame from the results
df_scores = pd.DataFrame(results)

In [ ]:
df_scores.sort_values(by="f1_score", ascending=False).to_clipboard()


In [ ]:
# Load the JSON file
with open(r"C:\Users\neosw\Desktop\WL\V6_unsupervised_sequences\_ae_results\23m_11163_shift__summary.json", "r") as f:
    best_lstm_ae_results = json.load(f)

# Display the parameters
from pprint import pprint

pprint(best_lstm_ae_results["best_params"])


In [ ]:
best_model_name = best_lstm_ae_results["checkpoint"]


In [ ]:
# Load the JSON file
with open(r"C:\Users\neosw\Desktop\WL\V6_unsupervised_sequences\_ae_results\23m_11163_shift__grid_logs.json", "r") as f:
    grid_logs = json.load(f)


In [ ]:
from pprint import pprint

# Filter the logs for the best model
best_model_logs = next((log for log in grid_logs if log["checkpoint"] == best_model_name), None)

# Display the logs
if best_model_logs:
    pprint(best_model_logs)
else:
    print("No logs found for the best model.")

In [ ]:
import matplotlib.pyplot as plt

# Extract epoch results
epoch_results = best_model_logs["epoch_results"]

# Extract epochs, train_loss, and val_loss
epochs = [result["epoch"] for result in epoch_results]
train_loss = [result["train_loss"] for result in epoch_results]
val_loss = [result["val_loss"] for result in epoch_results]

# Plot the losses
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
ax.plot(epochs, train_loss, label="Train Loss", marker="")
ax.plot(epochs, val_loss, label="Validation Loss", marker="")
ax.set_xlabel("Epochs")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.7)

# Hide top and right spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()


In [ ]:
best_model_name


In [ ]:
# --- Auto-infer LSTM-AE architecture from a checkpoint, load strictly, and summarize ---

import json
import os

import torch
import torch.nn as nn

checkpoint_path = rf"C:\Users\neosw\Desktop\WL\{best_model_name}"  # your path


# ----- Model def (same as training) -----
class LSTMAE(nn.Module):
    def __init__(self, input_dim, enc_hidden, enc_layers, latent_dim, dec_hidden, dec_layers, dropout=0.0):
        super().__init__()
        self.input_dim = input_dim
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=enc_hidden,
            num_layers=enc_layers,
            batch_first=True,
            dropout=dropout if enc_layers > 1 else 0.0,
        )
        self.to_latent = nn.Sequential(nn.Linear(enc_hidden, latent_dim), nn.ReLU(inplace=True))
        self.decoder = nn.LSTM(
            input_size=latent_dim,
            hidden_size=dec_hidden,
            num_layers=dec_layers,
            batch_first=True,
            dropout=dropout if dec_layers > 1 else 0.0,
        )
        self.to_output = nn.Linear(dec_hidden, input_dim)

    def forward(self, x):  # x: [B, T, C]
        B, T, C = x.shape
        _, (h_n, _) = self.encoder(x)  # h_n: [enc_layers, B, enc_hidden]
        h_last = h_n[-1]  # [B, enc_hidden]
        z = self.to_latent(h_last)  # [B, latent_dim]
        z_rep = z.unsqueeze(1).repeat(1, T, 1)
        dec_out, _ = self.decoder(z_rep)
        y_hat = self.to_output(dec_out)  # [B, T, C]
        return y_hat


def infer_arch_from_state(state_dict: dict):
    """
    Infer (input_dim, enc_hidden, enc_layers, latent_dim, dec_hidden, dec_layers)
    from LSTM/Linear weight shapes in a saved state_dict.
    """
    # --- Encoder ---
    # encoder.weight_hh_l0 shape = [4*H_enc, H_enc]
    w_hh_l0 = state_dict["encoder.weight_hh_l0"]
    fourH_enc, H_enc = w_hh_l0.shape
    assert fourH_enc % 4 == 0, "encoder.weight_hh_l0 rows not multiple of 4"
    enc_hidden = H_enc
    # Count encoder layers by probing weight_ih_lk keys
    enc_layers = 0
    while f"encoder.weight_ih_l{enc_layers}" in state_dict:
        enc_layers += 1
    if enc_layers == 0:
        raise ValueError("Cannot find encoder layers in state_dict")

    # --- Bottleneck (to_latent) ---
    # to_latent.0.weight shape = [latent_dim, enc_hidden]
    W_lat = state_dict["to_latent.0.weight"]
    latent_dim, enc_hidden_check = W_lat.shape
    assert enc_hidden_check == enc_hidden, "to_latent input dim != enc_hidden"

    # --- Decoder ---
    # decoder.weight_hh_l0 shape = [4*H_dec, H_dec]
    w_hh_dec = state_dict["decoder.weight_hh_l0"]
    fourH_dec, H_dec = w_hh_dec.shape
    assert fourH_dec % 4 == 0, "decoder.weight_hh_l0 rows not multiple of 4"
    dec_hidden = H_dec
    # decoder.weight_ih_l0 shape = [4*H_dec, latent_dim]
    w_ih_dec = state_dict["decoder.weight_ih_l0"]
    _, latent_dim_check = w_ih_dec.shape
    assert latent_dim_check == latent_dim, "decoder input size != latent_dim"
    # Count decoder layers similarly
    dec_layers = 0
    while f"decoder.weight_ih_l{dec_layers}" in state_dict:
        dec_layers += 1
    if dec_layers == 0:
        dec_layers = 1  # fallback

    # --- Output head ---
    # to_output.weight shape = [input_dim, dec_hidden]
    W_out = state_dict["to_output.weight"]
    input_dim, dec_hidden_check = W_out.shape
    assert dec_hidden_check == dec_hidden, "to_output input dim != dec_hidden"

    return dict(
        input_dim=int(input_dim),
        enc_hidden=int(enc_hidden),
        enc_layers=int(enc_layers),
        latent_dim=int(latent_dim),
        dec_hidden=int(dec_hidden),
        dec_layers=int(dec_layers),
        dropout=0.0,  # not stored in state_dict; assume 0.0
    )


# ----- Load checkpoint and infer -> rebuild model -----
device = "cuda" if torch.cuda.is_available() else "cpu"
state = torch.load(checkpoint_path, map_location=device)
state_dict = state["state_dict"] if isinstance(state, dict) and "state_dict" in state else state

arch = infer_arch_from_state(state_dict)
print("Inferred architecture:", arch)

model = LSTMAE(**arch).to(device)
model.load_state_dict(state_dict, strict=True)
model.eval()
print("Checkpoint loaded OK with strict=True")

# ----- Pretty summary (torchinfo) -----
try:
    from torchinfo import summary

    # Pick a plausible T for the summary table; if you know it, set it directly.
    # This doesn't affect weights or loading; it's just for display.
    T_DISPLAY = 300  # e.g., 5 min at 1 Hz
    print(
        summary(
            model,
            input_size=(8, T_DISPLAY, arch["input_dim"]),  # (batch, T, C)
            device=device,
            col_names=("input_size", "output_size", "num_params", "trainable"),
            depth=4,
        )
    )
except Exception:
    print("torchinfo not installed; run `pip install torchinfo` for a nicer table.")
    print(model)


# ----- Param count + quick forward sanity -----
def count_params(m: nn.Module):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


print(f"Trainable params: {count_params(model):,}")

with torch.no_grad():
    xb = torch.randn(4, T_DISPLAY, arch["input_dim"], device=device)
    yb = model(xb)
print(f"Sanity forward: in {tuple(xb.shape)} -> out {tuple(yb.shape)}")


In [ ]:
# Load the JSON file
with open(r"C:\Users\neosw\Desktop\WL\V6_unsupervised_sequences\_ae_results\23m_11163_shift__scores.json", "r") as f:
    scores = json.load(f)


In [ ]:
thr = 0.6
best_cutoff = 0.048571
test_scores = scores["test"]["scores"]
true_labels = scores["test"]["labels"]

In [ ]:
un_test_data = read_features_parquet(
    r"C:\Users\neosw\Desktop\WL\V6_unsupervised_sequences\23m_11163_shift\test.parquet"
).to_pandas()

un_test_data["sanity_label"] = true_labels
un_test_data["predicted_score"] = low_pass_filter(np.array(test_scores), best_cutoff, fs=1)


In [ ]:
# Check for mismatched labels
mismatched_labels = un_test_data[un_test_data["label_shift"] != un_test_data["sanity_label"]]

# Display the mismatched rows, if any
if not mismatched_labels.empty:
    print("Mismatched labels found:")
    print(mismatched_labels)
else:
    print("No mismatched labels found.")

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

# Binarize the predicted_score column using the threshold
un_test_data["predicted_label"] = (un_test_data["predicted_score"] >= thr).astype(int)

# Calculate metrics
f1 = f1_score(un_test_data["label_shift"], un_test_data["predicted_label"])
precision = precision_score(un_test_data["label_shift"], un_test_data["predicted_label"])
recall = recall_score(un_test_data["label_shift"], un_test_data["predicted_label"])
conf_matrix = confusion_matrix(un_test_data["label_shift"], un_test_data["predicted_label"])

# Display the results
print(f"F1 Score: {f1}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print("Confusion Matrix:")
print(conf_matrix)


In [ ]:
from datetime import timedelta

# Initialize counters
detected_failures = 0
false_alarms = 0
thr = 0.6
# Define the cutoff date
cutoff_date = pd.Timestamp("2025-12-31")

# order un_test_data by window_end
un_test_data = un_test_data.sort_values(by="window_end")

# Iterate through the failure data
for _, failure in failure_data.iterrows():
    failure_start = failure["Start"]
    failure_end = failure["End"]

    # Skip failures after the cutoff date
    if failure_start > cutoff_date:
        continue

    # Define the 4-day window around the failure
    window_start = failure_start - timedelta(days=6)
    window_end = failure_start - timedelta(days=1)

    # Filter test_data within the 4-day window
    window_data = un_test_data[
        (un_test_data["window_end"] >= window_start) & (un_test_data["window_end"] <= window_end)
    ]

    # Check for continuous crossings above the threshold
    above_threshold = window_data["predicted_score"] >= thr
    if above_threshold.any():
        # Count as detected if it stays above the threshold at least once
        detected_failures += 1

# Check for false alarms (crossings outside of failure windows)
for idx, row in un_test_data.iterrows():
    timestamp_end = row["window_end"]
    timestamp_start = row["timestamp"]
    predicted_score = row["predicted_score"]

    # Skip rows after the cutoff date
    if timestamp_end > cutoff_date:
        continue

    # If the predicted_score crosses the threshold
    if predicted_score >= thr:
        # Check if the timestamp is within any failure window
        in_failure_window = any(
            (timestamp_start <= failure["End"] and timestamp_end >= failure["Start"] - timedelta(days=6))
            for _, failure in failure_data.iterrows()
        )

        # If not in a failure window, check for crossing behavior
        if not in_failure_window:
            # Check if the previous score was below the threshold using the index
            if idx > 0:  # Ensure there is a previous row
                previous_row = un_test_data.iloc[idx - 1]
                if previous_row["predicted_score"] < thr:
                    false_alarms += 1

# Display the results
print(f"Detected Failures: {detected_failures}")
print(f"False Alarms: {false_alarms}")


In [ ]:
from datetime import timedelta

# Initialize counters
detected_failures = 0
false_alarms = 0
thr = 0.6
# Define the cutoff date
cutoff_date = pd.Timestamp("2025-04-15")

# order un_test_data by window_end
un_test_data = un_test_data.sort_values(by="window_end")

# Iterate through the failure data
for _, failure in failure_data.iterrows():
    failure_start = failure["Start"]
    failure_end = failure["End"]

    # Skip failures after the cutoff date
    if failure_start > cutoff_date:
        continue

    # Define the 4-day window around the failure
    window_start = failure_start - timedelta(days=6)
    window_end = failure_start - timedelta(days=1)

    # Filter test_data within the 4-day window
    window_data = un_test_data[
        (un_test_data["window_end"] >= window_start) & (un_test_data["window_end"] <= window_end)
    ]

    # Check for continuous crossings above the threshold
    above_threshold = window_data["predicted_score"] >= thr
    if above_threshold.any():
        # Count as detected if it stays above the threshold at least once
        detected_failures += 1

# Check for false alarms (crossings outside of failure windows)
for idx, row in un_test_data.iterrows():
    timestamp_end = row["window_end"]
    timestamp_start = row["timestamp"]
    predicted_score = row["predicted_score"]

    # Skip rows after the cutoff date
    if timestamp_end > cutoff_date:
        continue

    # If the predicted_score crosses the threshold
    if predicted_score >= thr:
        # Check if the timestamp is within any failure window
        in_failure_window = any(
            (timestamp_start <= failure["End"] and timestamp_end >= failure["Start"] - timedelta(days=6))
            for _, failure in failure_data.iterrows()
        )

        # If not in a failure window, check for crossing behavior
        if not in_failure_window:
            # Check if the previous score was below the threshold using the index
            if idx > 0:  # Ensure there is a previous row
                previous_row = un_test_data.iloc[idx - 1]
                if previous_row["predicted_score"] < thr:
                    false_alarms += 1

# Display the results
print(f"Detected Failures: {detected_failures}")
print(f"False Alarms: {false_alarms}")


In [ ]:
# Define the cutoff date
cutoff_date = pd.Timestamp("2025-04-15")

# Filter failures before the cutoff date
failures_before_cutoff = failure_data[failure_data["Start"] < cutoff_date]

# Count the number of failures
num_failures = len(failures_before_cutoff)

print(f"Number of failures before {cutoff_date}: {num_failures}")

In [ ]:
plot_predicted_scores(un_test_data, (datetime(2025, 2, 10), datetime(2025, 6, 2)), thr)


In [ ]:
for failure_start, subsystem in zip(failure_data["Start"], failure_data["Subsystem Bezeichner"], strict=False):
    print(f"Subsystem Bezeichner: {subsystem}")
    start_xlim = failure_start - pd.Timedelta(days=6)
    end_xlim = failure_start + pd.Timedelta(days=1)
    plot_predicted_scores(un_test_data, (start_xlim, end_xlim), thr)


In [ ]:
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np


def plot_normalized_scores(
    sup_data: pd.DataFrame,
    un_data: pd.DataFrame,
    sup_threshold: float,
    unsup_threshold: float,
    xlim: tuple[datetime, datetime],
) -> None:
    """
    Plots normalized supervised and unsupervised scores over time with thresholds.

    Parameters:
    sup_data (pd.DataFrame): DataFrame containing supervised test data.
    un_data (pd.DataFrame): DataFrame containing unsupervised test data.
    sup_threshold (float): Threshold for supervised scores.
    unsup_threshold (float): Threshold for unsupervised scores.
    xlim (tuple): A tuple of two datetime objects specifying the x-axis limits.
    """
    # Extract scores
    sup_scores = sup_data["predicted_score"].values
    un_scores = un_data["predicted_score"].values

    # Normalize the supervised and unsupervised scores
    sup_scores_norm = sup_scores / sup_threshold
    un_scores_norm = un_scores / unsup_threshold

    # Normalize the thresholds
    sup_threshold_norm = 1.0

    # Create the plot
    fig, ax = plt.subplots(figsize=(12, 6), dpi=300)

    # Plot normalized supervised scores
    ax.plot(
        sup_data["window_end"],
        sup_scores_norm,
        label="Supervised Scores",
        color="blue",
    )

    # Plot normalized unsupervised scores
    ax.plot(
        un_data["window_end"],
        un_scores_norm,
        label="Unsupervised Scores",
        color="orange",
    )

    # Add horizontal lines for thresholds
    ax.axhline(y=sup_threshold_norm, color="black", linestyle="--", label="Normalized Threshold")

    # Format x-axis as time
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    # ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    # plt.xticks(rotation=45)

    # Add shaded regions for failure zones
    for _, row in failure_data.iterrows():
        ax.axvline(x=row["Start"] - pd.Timedelta(days=1), color="green", linestyle="--", label="Time to fix Failure")
        ax.axvspan(row["Start"], row["End"], color="red", alpha=0.3, label="Failure Zone")

    # Remove duplicate labels in the legend
    handles, labels = ax.get_legend_handles_labels()
    unique_labels = dict(zip(labels, handles, strict=False))
    ax.legend(
        unique_labels.values(),
        unique_labels.keys(),
        loc="lower center",
        bbox_to_anchor=(0.5, -0.2),
        ncol=3,
        frameon=False,
    )

    # Set x-axis limits
    ax.set_xlim(xlim)

    # Remove top and right spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Add labels, legend, and grid
    ax.set_xlabel("")
    ax.set_ylabel("Normalized Scores")
    plt.grid(axis="y", linestyle="--", alpha=0.7)

    # Show the plot
    plt.tight_layout()
    plt.show()


In [ ]:
plot_normalized_scores(
    sup_data=sup_test_data,
    un_data=un_test_data,
    sup_threshold=0.15,
    unsup_threshold=0.6,
    xlim=(datetime(2025, 2, 10), datetime(2025, 6, 2)),
)

In [ ]:
for failure_start, subsystem in zip(failure_data["Start"], failure_data["Subsystem Bezeichner"], strict=False):
    print(f"Subsystem Bezeichner: {subsystem}")
    start_xlim = failure_start - pd.Timedelta(days=6)
    end_xlim = failure_start + pd.Timedelta(days=1)
    plot_normalized_scores(
        sup_data=sup_test_data,
        un_data=un_test_data,
        sup_threshold=0.15,
        unsup_threshold=0.6,
        xlim=(start_xlim, end_xlim),
    )

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

# Binarize the predicted_score column using the threshold
un_test_data["predicted_label"] = (un_test_data["predicted_score"] >= thr).astype(int)

# Calculate metrics
f1 = f1_score(un_test_data["label_shift"], un_test_data["predicted_label"])
precision = precision_score(un_test_data["label_shift"], un_test_data["predicted_label"])
recall = recall_score(un_test_data["label_shift"], un_test_data["predicted_label"])
conf_matrix = confusion_matrix(un_test_data["label_shift"], un_test_data["predicted_label"])

# Display the results
print(f"F1 Score: {f1}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print("Confusion Matrix:")
print(conf_matrix)
